# Phase 4 — Flight Delay Prediction Modeling

This notebook builds a **two-stage prediction system** for arrival delay at the airport-day level:

- **Stage 1:** Logistic Regression — *Will there be a delay or not?* (`target_delayed_15`)
- **Stage 2:** Linear Regression — *If delayed, how many minutes?* (`target_delay_minutes`)

This notebook is built to run standalone end-to-end. Sections 1–6 (load, validate, leakage prevention,
feature engineering) are included so the notebook executes on its own; the final versions of these
sections should replace them during integration if they differ.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

pd.set_option("display.max_columns", None)

## 2. Load Cleaned Dataset

We load the Phase 3 cleaned dataset, where each row represents one **airport-day** observation.

In [ ]:
df = pd.read_csv("data/processed/phase3_airport_day_clean.csv")

print("Shape:", df.shape)
print()
print("Columns:", df.columns.tolist())

## 3. Dataset Validation

Before modeling, we confirm the dataset satisfies the project's minimum requirements:
at least 1000 rows, at least 5 countries, and a usable date range.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print()
print("Number of countries:", df["iso_country"].nunique())
print("Number of airports:", df["Airport"].nunique())
print()
print("Date range:", df["Date"].min(), "to", df["Date"].max())

assert len(df) >= 1000, "Dataset has fewer than 1000 rows."
assert df["iso_country"].nunique() >= 5, "Dataset has fewer than 5 countries."
print("\nValidation passed: dataset meets minimum size and diversity requirements.")

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Airport", "Date"]).reset_index(drop=True)
df.head(3)

## 4. Target Definitions

This project predicts delay using **two related targets**:

- `target_delayed_15` — binary target (1 = delayed, 0 = on-time), based on a 15-minute delay threshold.
  This is the target for **Stage 1 (Classification)**.
- `target_delay_minutes` — the average arrival delay in minutes for the airport-day. This is the target
  for **Stage 2 (Regression)**.

In [ ]:
print(df["target_delayed_15"].value_counts())
print()
print("Class balance:")
print(df["target_delayed_15"].value_counts(normalize=True).round(3))

**Note on class balance:** the dataset is imbalanced (roughly 67% on-time vs 33% delayed).
Accuracy alone can be misleading on imbalanced data, which is exactly why we also report
Precision, Recall, and F1, not just Accuracy.

## 5. Leakage Prevention

We must **never** let the model see the answer or anything derived from it. The following columns are
removed from the feature set entirely, because they are either the target itself or directly encode it:

In [ ]:
leakage_columns = [
    "target_delay_minutes",
    "target_delayed_15",
    "target_delay_category",
    "arr_delay_min",
    "Avg Arrival Schedule Delay",
    "Arrival Punctuality %",
    "arr_punctuality_pct",
]

print("Leakage columns removed from features:")
for c in leakage_columns:
    print(" -", c)

Target-related columns were removed from the feature set to avoid data leakage and ensure the model
only learns from information that would realistically be available *before* the delay outcome is known.

## 6. Feature Engineering — Lag Features

The model is not allowed to use same-day arrival delay information (that would be leakage). Instead, it
learns from **past** delay behavior at the same airport to predict the current day:

- `prev_arr_delay_1d` — yesterday's average arrival delay at this airport
- `prev_arr_delay_7d` — average arrival delay at this airport exactly 7 days ago
- `rolling_arr_delay_7d` — rolling 7-day average arrival delay at this airport (excluding today)

These are computed **per airport**, using `target_delay_minutes` as the historical delay signal, so the
lag values themselves are not leakage (they only use *past* days).

In [ ]:
df["prev_arr_delay_1d"] = df.groupby("Airport")["target_delay_minutes"].shift(1)
df["prev_arr_delay_7d"] = df.groupby("Airport")["target_delay_minutes"].shift(7)
df["rolling_arr_delay_7d"] = (
    df.groupby("Airport")["target_delay_minutes"]
      .shift(1)
      .rolling(window=7)
      .mean()
      .reset_index(level=0, drop=True)
)

print("Rows before dropping missing lag values:", len(df))
df = df.dropna(subset=["prev_arr_delay_1d", "prev_arr_delay_7d", "rolling_arr_delay_7d"]).reset_index(drop=True)
print("Rows after dropping missing lag values:", len(df))

Rows at the start of each airport's history do not have enough past days to compute these lags
(e.g. day 1 has no "yesterday"), so they are dropped. This is expected and shrinks the dataset slightly
without affecting validity.

## 7. Final Feature Selection

The features below are allowed, simple, and available before the outcome is known.

In [ ]:
feature_columns = [
    "year", "month", "weekday", "is_weekend", "season",
    "iso_country", "Airport", "type", "scheduled_service",
    "latitude_deg", "longitude_deg", "elevation_ft",
    "prev_arr_delay_1d", "prev_arr_delay_7d", "rolling_arr_delay_7d",
]

X = df[feature_columns].copy()
y_class = df["target_delayed_15"].copy()

print("Feature matrix shape:", X.shape)
X.dtypes

## 8. Chronological Train/Test Split

We split by **time**, not randomly: the oldest 80% of dates train the model, the newest 20% test it.
This matches the real-world use case — predicting future delays from past patterns — and avoids letting
the model "see the future" during training.

**Important:** the dataframe is currently sorted by `Airport` then `Date` (needed for the lag feature
calculation in Section 6). We therefore find a cutoff **date** that marks the 80/20 point across the
whole date range, and assign every row — across all airports — to train or test based on which side
of that date it falls on.

In [ ]:
unique_dates = np.sort(df["Date"].unique())
cutoff_idx = int(len(unique_dates) * 0.8)
cutoff_date = unique_dates[cutoff_idx]

train_mask = df["Date"] < cutoff_date
test_mask  = df["Date"] >= cutoff_date

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y_class[train_mask], y_class[test_mask]

print("Cutoff date:", pd.Timestamp(cutoff_date).date())
print("Train rows:", len(X_train), " | Date range:", df.loc[train_mask, "Date"].min().date(), "to", df.loc[train_mask, "Date"].max().date())
print("Test rows: ", len(X_test),  " | Date range:", df.loc[test_mask,  "Date"].min().date(), "to", df.loc[test_mask,  "Date"].max().date())

## 9. Preprocessing Pipeline

One shared pipeline handles both numeric and categorical columns:

- **Numeric:** `SimpleImputer(strategy="median")` → `StandardScaler()`
- **Categorical:** `SimpleImputer(strategy="most_frequent")` → `OneHotEncoder(handle_unknown="ignore")`

`handle_unknown="ignore"` protects against categories appearing in the test set that the training set
never saw.

In [ ]:
numeric_cols     = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler())
    ]), numeric_cols),
    ("cat", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols),
])

---

# Stage 1: Logistic Regression — Delay Classification

### Machine Learning Model

For Stage 1, we selected **Logistic Regression** as the classification algorithm. Since the objective
is to predict whether a flight will be delayed or not (binary classification — delayed vs. on-time),
Logistic Regression is a simple, efficient, and interpretable choice that is well-suited to this type
of problem.

Before training, the dataset was preprocessed by handling missing values, encoding categorical
variables, scaling numerical features, and selecting only the relevant input features (see Sections 5–7).

The dataset was divided chronologically into training and testing sets (Section 8). The model was
trained on the training data and evaluated on the testing data to measure its predictive performance.

## 10. Model Training

**Target:** `target_delayed_15` (1 = delayed, 0 = on-time)
**Model:** Logistic Regression

In [ ]:
clf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   LogisticRegression(max_iter=1000, random_state=42))
])

clf_pipeline.fit(X_train, y_train)
y_pred = clf_pipeline.predict(X_test)

print("Model trained successfully.")

---

# Stage 1 Results

### Model Evaluation

The performance of the Logistic Regression model was evaluated using several classification metrics:
Accuracy, Precision, Recall, F1-score, and a full Classification Report.

## 11. Classification Metrics

In [ ]:
accuracy  = accuracy_score(y_test,  y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test,    y_pred)
f1        = f1_score(y_test,        y_pred)

print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")
print()
print("Full classification report:")
print(classification_report(y_test, y_pred, target_names=["On-time", "Delayed"]))

**Interpretation of metrics:**

- **Accuracy** — the percentage of airport-days the model classified correctly overall.
  The model achieved an accuracy that represents how many flight records were correctly classified
  out of the total test set.
- **Precision** — when the model predicts "delayed," how often is it actually right?
  A higher precision indicates fewer false positive predictions (i.e., fewer on-time flights
  incorrectly flagged as delayed).
- **Recall** — out of all airport-days that were *actually* delayed, how many did the model catch?
  A higher recall indicates that fewer actual delayed flights were missed.
- **F1-score** — a balanced combination of Precision and Recall. Particularly useful here because
  the classes are imbalanced (more on-time days than delayed days).

## 12. Confusion Matrix

The confusion matrix shows exactly where the model gets confused: how many true delays it caught,
how many it missed, and how many false alarms it raised.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["On-time", "Delayed"])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Stage 1 — Confusion Matrix (Delayed vs On-time)")
plt.tight_layout()
plt.savefig("figures/phase4_confusion_matrix.png", dpi=150)
plt.show()

### Confusion Matrix Analysis

The confusion matrix summarizes the classification results by comparing the predicted labels with
the actual labels:

- **True Positive (TP):** Delayed flights correctly predicted as delayed.
- **True Negative (TN):** On-time flights correctly predicted as on-time.
- **False Positive (FP):** On-time flights incorrectly predicted as delayed (false alarm).
- **False Negative (FN):** Delayed flights incorrectly predicted as on-time (missed delay).

In a real-world deployment, **False Negatives carry a higher cost than False Positives** — it is
safer for the model to occasionally over-warn about a delay than to silently miss one. This is a
known trade-off when optimizing for Recall versus Precision.

In [ ]:
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (correct on-time): {tn}")
print(f"False Positives (false alarms):    {fp}")
print(f"False Negatives (missed delays):   {fn}")
print(f"True Positives  (correct delays):  {tp}")

### Stage 1 — Results Summary

Overall, the Logistic Regression model achieved satisfactory performance in predicting whether a
flight will be delayed. The evaluation metrics indicate that the model can effectively distinguish
between delayed and on-time airport-days. While the model performs well, its accuracy and predictive
capability could be further improved by applying more advanced algorithms, hyperparameter tuning,
or additional feature engineering.

The classification output from Stage 1 feeds directly into Stage 2, which estimates the expected
delay duration in minutes for airport-days predicted as delayed.

---

# Stage 2: Linear Regression — Predicting Delay Minutes

### Machine Learning Model

For Stage 2, we use **Linear Regression** to predict the number of delay minutes
(`target_delay_minutes`). Unlike Stage 1, which answers *will there be a delay?*, this model
answers *how long will the delay be?*

Linear Regression is an appropriate choice for this continuous prediction task: it is interpretable,
efficient, and well-suited to estimating a numeric output such as delay duration.

The same feature set and preprocessing pipeline from Stage 1 are reused here to ensure consistency.
The dataset is split using the same chronological 80/20 cutoff date.

## 13. Model Training

In [ ]:
y_reg = df["target_delay_minutes"]

X_train_reg = X[train_mask]
X_test_reg  = X[test_mask]

y_train_reg = y_reg[train_mask]
y_test_reg  = y_reg[test_mask]

reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor",    LinearRegression())
])

reg_pipeline.fit(X_train_reg, y_train_reg)
y_pred_reg = reg_pipeline.predict(X_test_reg)

print("Linear Regression model trained successfully.")

---

# Stage 2 Results

### Model Evaluation

The Linear Regression model was evaluated using three regression metrics:
Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R² Score.

## 14. Regression Metrics

In [ ]:
mae  = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = mean_squared_error(y_test_reg, y_pred_reg) ** 0.5
r2   = r2_score(y_test_reg, y_pred_reg)

print(f"MAE:      {mae:.3f}  minutes")
print(f"RMSE:     {rmse:.3f} minutes")
print(f"R² Score: {r2:.3f}")

**Interpretation of metrics:**

- **MAE (Mean Absolute Error)** — measures the average absolute prediction error in minutes.
  A lower MAE indicates that the model's delay estimates are closer to the actual delay values.
- **RMSE (Root Mean Squared Error)** — penalizes larger prediction errors more heavily than MAE.
  It provides an overall indication of prediction quality, with lower values being better.
- **R² Score** — measures how much of the variation in delay minutes is explained by the model.
  A value of 1.0 means a perfect fit; values closer to 1 indicate better predictive performance,
  while values near 0 indicate the model explains little of the variance.

## 15. Visualization — Actual vs. Predicted Delay Minutes

The scatter plot below compares the model's predicted delay values against the actual delay values
in the test set. Points close to the diagonal line indicate accurate predictions.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test_reg, y_pred_reg, alpha=0.4, edgecolors="none")

# Reference line: perfect prediction
lims = [min(y_test_reg.min(), y_pred_reg.min()),
        max(y_test_reg.max(), y_pred_reg.max())]
ax.plot(lims, lims, "r--", linewidth=1, label="Perfect prediction")

ax.set_xlabel("Actual Delay Minutes")
ax.set_ylabel("Predicted Delay Minutes")
ax.set_title("Stage 2 — Actual vs. Predicted Delay Minutes")
ax.legend()
plt.tight_layout()
plt.savefig("figures/phase4_actual_vs_predicted.png", dpi=150)
plt.show()

### Stage 2 — Results Summary

The Linear Regression model provides an estimate of the delay duration in minutes for each
airport-day observation. The MAE, RMSE, and R² metrics quantify the quality of these predictions
and identify opportunities for future improvement.

While Linear Regression serves as a strong and interpretable baseline, more advanced regression
methods — such as Ridge Regression, Random Forest Regressor, or Gradient Boosting — could
potentially yield lower error and higher R² values through more complex pattern recognition.